# FarmFederate — Weather-Modality Training (Colab GPU)

Trains the RoBERTa + ViT crop-stress model with the **AMFU Kharagpur (IMD 42893)** weather modality,
and runs the `full` / `text-only` / `none` ablation.

**Before running:** `Runtime → Change runtime type → GPU` (T4 is enough), then `Runtime → Run all`.


## 1. Check GPU


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available(), 'No GPU — set Runtime type to GPU and Run all again.'
print('torch', torch.__version__, '| CUDA device:', torch.cuda.get_device_name(0))


## 2. Get the code (clones `feature/multimodal-work`)
If the repo is private, you'll be prompted for a GitHub token (a fine-grained PAT with read access).


In [ ]:
import subprocess, getpass, os
REPO   = 'ayushdebnath012/FarmFederate-Advisor'
BRANCH = 'feature/multimodal-work'
if os.path.isdir('repo'):
    subprocess.run(['rm','-rf','repo'])
def clone(url):
    return subprocess.run(['git','clone','--depth','1','--branch',BRANCH,url,'repo'],capture_output=True,text=True)
r = clone(f'https://github.com/{REPO}.git')
if r.returncode != 0:
    print('Public clone failed (repo is likely private).')
    tok = getpass.getpass('GitHub token: ')
    r = clone(f'https://{tok}@github.com/{REPO}.git')
assert r.returncode == 0, r.stderr
print('cloned; HEAD:')
print(subprocess.run(['git','-C','repo','log','--oneline','-1'],capture_output=True,text=True).stdout)


## 3. Install dependencies
Colab already ships torch, numpy, pandas, scikit-learn and Pillow; we add the NLP/vision + Excel bits.


In [ ]:
!pip -q install 'transformers>=4.30' 'datasets>=2.12' tokenizers openpyxl 2>/dev/null
print('deps ready')


## 4. Sanity-check the weather parser
Confirms the workbook loads and the agromet rule-labels have healthy support.


In [ ]:
%cd /content/repo/backend
!python weather_data.py | tail -4


## 5. (optional) Mount Drive to keep checkpoints
Skip this cell to keep outputs only in the Colab session.


In [ ]:
USE_DRIVE = False  # set True to persist to Drive
OUT = '/content/checkpoints'
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/FarmFederate/checkpoints'
import os; os.makedirs(OUT, exist_ok=True); print('checkpoints ->', OUT)


## 6. Train — full weather modality
Unfrozen backbones, real plant-stress images (auto-downloaded), mixed precision.
Adjust `--epochs`, `--max-samples`, `--max-images` to trade speed for quality.


In [ ]:
!python -u multimodal_train.py \
    --weather-mode full \
    --epochs 6 --batch-size 32 --lr 3e-5 --amp \
    --max-samples 6000 --max-per-source 2000 \
    --max-images 4000 --max-per-image-dataset 2000 \
    --out $OUT --tag colab


## 7. Ablation — `text-only` and `none`
Same recipe, different use of the station data, for the weather-vs-no-weather comparison.
Comment this out if you only need the `full` model.


In [ ]:
for MODE in ['text-only','none']:
    print('='*30, MODE, '='*30)
    !python -u multimodal_train.py --weather-mode {MODE} --epochs 6 --batch-size 32 --lr 3e-5 --amp \
        --max-samples 6000 --max-per-source 2000 --max-images 4000 --max-per-image-dataset 2000 --out $OUT --tag colab


## 8. Compare the runs


In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob(f'{OUT}/central_*_colab/metrics.json')):
    d = json.load(open(f)); v = d['best_val']
    rows.append({'mode': d['weather_mode'], 'best_epoch': d['best_epoch'],
                 'f1_macro': round(v['f1_macro'],4), 'f1_micro': round(v['f1_micro'],4),
                 **{k: round(x,3) for k,x in v['f1_per_label'].items()}})
pd.DataFrame(rows).set_index('mode') if rows else print('no metrics yet')


## 9. (optional) Download the best `full` checkpoint


In [ ]:
from google.colab import files
ckpt = f'{OUT}/central_full_colab/global_central.pt'
import os; files.download(ckpt) if os.path.exists(ckpt) else print('checkpoint not found:', ckpt)
